[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/Advanced-Time-Series/blob/main/Quantlets/Ch_16/ATS_ch16_rational_bubbles/ATS_ch16_rational_bubbles.ipynb)

# ATS_ch16_rational_bubbles

Rational bubbles: a Blanchard-Watson bubble that can burst (and turn negative, the case excluded by Diba and Grossman), an Evans (1991) periodically collapsing bubble added to the present-value fundamental of random-walk dividends; whole-sample ADF against SADF, GSADF and BSADF date-stamping with Monte Carlo critical values.

*Advanced Time Series Analysis and Forecasting (ATS), Chapter 16: Explosive roots and bubbles. Author: Daniel Traian Pele.*

> **Working in Google Colab? Save your own copy first.**
> The Colab link opens a temporary copy: when you close the tab or the session times out, your changes are lost.
> Use **File → Save a copy in Drive** (it goes to the *Colab Notebooks* folder of your ASE Google Drive) and work in that copy.
> For the team project, **File → Save a copy in GitHub** puts the notebook directly in your team's repository.
> Files you create while the notebook runs (CSV, charts) are also temporary: set `SAVE_TO_DRIVE = True` in the next cell to keep them.

In [ ]:
# Optional (Colab only): keep the files you create in Google Drive
SAVE_TO_DRIVE = False   # set to True, run this cell and allow access to your Drive

import os
OUT_DIR = '.'
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/ATS/Chapter_16'
    os.makedirs(OUT_DIR, exist_ok=True)
print('Save your outputs to:', os.path.abspath(OUT_DIR))


In [ ]:
# the residual unit-root test of the LPPLS filter uses arch (Phillips-Perron)
%pip install -q arch

In [ ]:
import os
import re
import json
import urllib.request
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgb
from scipy import optimize
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

## Chart style

In [ ]:
# Palette (RGB values of latex/preamble.tex)
MainBlue = '#1A3A6E'
IDAred = '#CD0000'
Forest = '#2E7D32'
Amber = '#B5853F'
Orange = '#E67E22'
Purple = '#8E44AD'
Teal = '#17A2B8'
Crimson = '#DC3545'
DarkText = '#1F2A44'          # text, axes and ticks (dark navy, not grey)
LightBlue = '#7EA6E0'         # thin background series (e.g. the other countries of a panel): a blue tint, never grey

PALETTE = [MainBlue, IDAred, Forest, Amber, Purple, Orange, Teal, Crimson]
# fixed colours for the series used in several chapters
COL = {'sp500': MainBlue, 'bet': IDAred, 'bettr': Orange, 'dax': Teal, 'btc': Amber, 'eth': Crimson,
       'eurron': Forest, 'gold': Purple, 'vix': Crimson, 'stoxx50': Forest, 'ndx': Teal}


def apply():
    """Set the course style for matplotlib."""
    rc = plt.rcParams
    rc['figure.facecolor'] = 'none'
    rc['axes.facecolor'] = 'none'
    rc['savefig.facecolor'] = 'none'
    rc['savefig.transparent'] = True
    rc['axes.grid'] = False
    rc['font.family'] = 'sans-serif'
    rc['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
    # font sizes for slides (charts are 9-11 inches wide and shown at 8-14 cm)
    rc['font.size'] = 12
    rc['axes.labelsize'] = 13
    rc['axes.titlesize'] = 13
    rc['xtick.labelsize'] = 11.5
    rc['ytick.labelsize'] = 11.5
    rc['legend.fontsize'] = 11
    rc['axes.spines.top'] = False
    rc['axes.spines.right'] = False
    rc['axes.linewidth'] = 0.6
    rc['lines.linewidth'] = 1.4
    rc['legend.facecolor'] = 'none'
    rc['legend.framealpha'] = 0
    rc['legend.frameon'] = False
    for k in ('text.color', 'axes.labelcolor', 'axes.edgecolor', 'xtick.color', 'ytick.color', 'axes.titlecolor'):
        rc[k] = DarkText
    rc['axes.prop_cycle'] = mpl.cycler(color=PALETTE)


def legend_outside_bottom(ax, ncol=2, y=-0.22, **kw):
    """Place the legend outside the plot, bottom centre (for a figure with several axes, pass the last one
    or use fig.legend with the same arguments)."""
    return ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False, **kw)


def fig_legend_bottom(fig, handles=None, labels=None, ncol=3, y=-0.02):
    """One legend for a whole figure (several panels), below the panels."""
    if handles is None:
        handles, labels = [], []
        for ax in fig.axes:
            for h, l in zip(*ax.get_legend_handles_labels()):
                if l not in labels and not l.startswith('_'):
                    handles.append(h)
                    labels.append(l)
    return fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


def _is_grey(c, tol=0.06):
    try:
        r, g, b = to_rgb(c)
    except ValueError:
        return False
    return max(r, g, b) - min(r, g, b) < tol and 0.25 < (r + g + b) / 3 < 0.95


def check_no_grey(fig):
    """House rule: no grey series and no grey text. Raises ValueError listing the offending elements."""
    bad = []
    for ax in fig.axes:
        for ln in ax.get_lines():
            if not ln.get_label().startswith('_') and _is_grey(ln.get_color()):
                bad.append(f'line {ln.get_label()!r}')
        for coll in ax.collections:
            fc = coll.get_facecolor()
            if len(fc) and coll.get_label() and not coll.get_label().startswith('_') and _is_grey(fc[0][:3]):
                bad.append(f'series {coll.get_label()!r}')
        for t in ax.texts + [ax.title, ax.xaxis.label, ax.yaxis.label]:
            if t.get_text() and _is_grey(t.get_color()):
                bad.append(f'text {t.get_text()[:30]!r}')
    if bad:
        raise ValueError('grey elements: ' + ', '.join(bad))


def save_fig(name, out_dir=None, show=True):
    """Save the figure as transparent PDF and PNG (OUT_DIR if defined, else the current folder), then show it."""
    d = out_dir or globals().get('OUT_DIR', '.')
    plt.savefig(os.path.join(d, f'{name}.pdf'), bbox_inches='tight', transparent=True)
    plt.savefig(os.path.join(d, f'{name}.png'), bbox_inches='tight', transparent=True, dpi=180)
    if show:
        plt.show()
    plt.close()


apply()
# the chapter scripts call the style as st.<name> (import ats_style as st): the same names here
import types
st = types.SimpleNamespace(COL=COL, PALETTE=PALETTE, MainBlue=MainBlue, IDAred=IDAred, Forest=Forest, Amber=Amber,
                           Orange=Orange, Purple=Purple, Teal=Teal, Crimson=Crimson, DarkText=DarkText, apply=apply, legend_outside_bottom=legend_outside_bottom,
                           fig_legend_bottom=fig_legend_bottom, save_fig=save_fig, check_no_grey=check_no_grey)

## Data

In [ ]:
REPO_RAW = 'https://raw.githubusercontent.com/danpele/Advanced-Time-Series/main/data/market/'
# local copy of the course data, if the notebook runs inside the repository
MARKET_DIR = next((os.path.join(d, 'data', 'market') for d in ('.', '..', '../..', '../../..')
                   if os.path.isdir(os.path.join(d, 'data', 'market'))), '')
# local copy of the Oxford-Man realized library, if present (not distributed; read_omi downloads it)
REALIZED_DIR = next((os.path.join(d, 'data', 'realized') for d in ('.', '..', '../..', '../../..')
                     if os.path.isdir(os.path.join(d, 'data', 'realized'))), '')
END = '2026-09-18'          # last day of the saved data
# Oxford-Man Institute's realized library, version 0.3 (Heber, Lunde, Shephard and Sheppard 2009): the last public
# file (28 February 2022), preserved by the Internet Archive; six indices and the main measures
OMI_URL = ('https://web.archive.org/web/20220301022212id_/'
           'https://realized.oxford-man.ox.ac.uk/images/oxfordmanrealizedvolatilityindices.zip')
OMI_FILE = 'omi_realized_library_v03.csv'
OMI_SYMBOLS = ['.SPX', '.GDAXI', '.FTSE', '.N225', '.STOXX50E', '.FCHI']
OMI_COLS = ['rv5', 'rv5_ss', 'rv10', 'bv', 'bv_ss', 'medrv', 'rk_parzen', 'rk_twoscale', 'rsv', 'open_to_close',
            'close_price', 'open_price', 'nobs']

# name -> (symbol, label, group, field, default start)
SERIES = {
    # equity indices
    'sp500':    ('GSPC.INDX',     'S&P 500',               'Equity', 'close', '2000-01-01'),
    'ndx':      ('NDX.INDX',      'Nasdaq 100',            'Equity', 'close', '2000-01-01'),
    'dax':      ('GDAXI.INDX',    'DAX',                   'Equity', 'close', '2000-01-01'),
    'stoxx50':  ('STOXX50E.INDX', 'Euro Stoxx 50',         'Equity', 'close', '2000-01-01'),
    'nikkei':   ('N225.INDX',     'Nikkei 225',            'Equity', 'close', '2000-01-01'),
    'bet':      ('BET',           'BET',                   'Equity', 'close', '2000-01-01'),
    'bettr':    ('BETTR.INDX',    'BET-TR',                'Equity', 'close', '2014-09-23'),
    'betxt':    ('BETXT.INDX',    'BET-XT',                'Equity', 'close', '2011-08-12'),
    'betfi':    ('BETFI.INDX',    'BET-FI',                'Equity', 'close', '2012-01-25'),
    'wig20':    ('WIG20.INDX',    'WIG20',                 'Equity', 'close', '2000-01-01'),
    'bux':      ('BUX.INDX',      'BUX',                   'Equity', 'close', '2000-01-01'),
    'px':       ('PX.INDX',       'PX',                    'Equity', 'close', '2000-01-01'),
    'vix':      ('VIX.INDX',      'VIX',                   'Volatility', 'close', '2000-01-01'),
    # ETFs (adjusted close)
    'spy':      ('SPY.US',        'SPY',                   'ETF', 'adjusted_close', '2000-01-01'),
    'tlt':      ('TLT.US',        'TLT (US Treasuries 20y+)', 'ETF', 'adjusted_close', '2002-07-30'),
    'gld':      ('GLD.US',        'GLD (gold ETF)',        'ETF', 'adjusted_close', '2004-11-18'),
    'tvbetetf': ('TVBETETF.RO',   'TVBETETF',              'ETF', 'adjusted_close', '2015-06-12'),
    # Bucharest Stock Exchange (adjusted close)
    'tlv':      ('TLV.RO',        'Banca Transilvania',    'Stock', 'adjusted_close', '2000-01-01'),
    'snp':      ('SNP.RO',        'OMV Petrom',            'Stock', 'adjusted_close', '2001-09-04'),
    'brd':      ('BRD.RO',        'BRD',                   'Stock', 'adjusted_close', '2001-01-16'),
    'tgn':      ('TGN.RO',        'Transgaz',              'Stock', 'adjusted_close', '2008-01-25'),
    'sng':      ('SNG.RO',        'Romgaz',                'Stock', 'adjusted_close', '2013-11-12'),
    'snn':      ('SNN.RO',        'Nuclearelectrica',      'Stock', 'adjusted_close', '2013-11-04'),
    'h2o':      ('H2O.RO',        'Hidroelectrica',        'Stock', 'adjusted_close', '2023-07-12'),
    # US stocks (adjusted close)
    'aapl':     ('AAPL.US',       'Apple',                 'Stock', 'adjusted_close', '2000-01-01'),
    'msft':     ('MSFT.US',       'Microsoft',             'Stock', 'adjusted_close', '2000-01-01'),
    'nvda':     ('NVDA.US',       'NVIDIA',                'Stock', 'adjusted_close', '2000-01-01'),
    'jpm':      ('JPM.US',        'JPMorgan Chase',        'Stock', 'adjusted_close', '2000-01-01'),
    # crypto (close, 7 days a week)
    'btc':      ('BTC-USD.CC',    'Bitcoin',               'Crypto', 'close', '2014-09-17'),
    'eth':      ('ETH-USD.CC',    'Ethereum',              'Crypto', 'close', '2015-08-07'),
    'sol':      ('SOL-USD.CC',    'Solana',                'Crypto', 'close', '2020-04-11'),
    'usdt':     ('USDT-USD.CC',   'Tether (USDT)',         'Crypto', 'close', '2015-02-26'),
    # FX and commodities
    'eurron':   ('REF:EUR',       'EUR/RON (BNR reference)', 'FX', 'close', '2005-07-01'),
    'usdron':   ('REF:USD',       'USD/RON (BNR reference)', 'FX', 'close', '2005-07-01'),
    'eurron_eodhd': ('EURRON.FOREX', 'EUR/RON (EODHD)',    'FX', 'close', '2005-07-01'),
    'eurusd':   ('EURUSD.FOREX',  'EUR/USD',               'FX', 'close', '2002-05-06'),
    'gold':     ('XAUUSD.FOREX',  'Gold (XAU/USD)',        'Commodity', 'close', '2000-01-01'),
    # government bond yields (close, % p.a.)
    'us10y':    ('US10Y.GBOND',   'US 10Y yield',          'Yield', 'close', '2000-01-01'),
    'de10y':    ('DE10Y.GBOND',   'Germany 10Y yield',     'Yield', 'close', '2007-03-28'),
    'ro10y':    ('RO10Y.GBOND',   'Romania 10Y yield',     'Yield', 'close', '2007-08-17'),
}
LABELS = {k: v[1] for k, v in SERIES.items()}
_CACHE = {}


def read_market(symbol):
    """Daily table of one symbol: local copy of the course data, otherwise the ATS repository on GitHub."""
    fname = f'{symbol}.csv'
    path = os.path.join(MARKET_DIR, fname) if MARKET_DIR else ''
    src = path if path and os.path.exists(path) else REPO_RAW + fname
    return pd.read_csv(src, index_col='date', parse_dates=True).sort_index()


def read_omi(symbols=None, local=True):
    """Oxford-Man Institute's realized library, version 0.3: daily realised measures (OMI_COLS) of six indices
    (OMI_SYMBOLS), columns date, symbol and the measures, sorted by symbol and date. The library is not
    redistributed with the course: a local copy data/realized/omi_realized_library_v03.csv is used if present,
    otherwise the last public file (Internet Archive copy of 28 February 2022) is downloaded once, the six indices
    are extracted and cached in the temporary folder. Cite: Heber, G., Lunde, A., Shephard, N. and Sheppard, K.
    (2009), Oxford-Man Institute's realized library, version 0.3, Oxford-Man Institute, University of Oxford."""
    import tempfile
    import zipfile
    if 'omi' not in _CACHE:
        cache = os.path.join(tempfile.gettempdir(), 'ats_omi')
        cands = ([os.path.join(REALIZED_DIR, OMI_FILE)] if (local and REALIZED_DIR) else []) + [os.path.join(cache, OMI_FILE)]
        path = next((c for c in cands if os.path.exists(c)), '')
        if not path:
            print("Downloading the Oxford-Man Institute's realized library, version 0.3 (Heber, Lunde, Shephard and "
                  "Sheppard 2009), last public file of 28 February 2022, from the Internet Archive")
            os.makedirs(cache, exist_ok=True)
            zpath = os.path.join(cache, 'oxfordmanrealizedvolatilityindices.zip')
            if not os.path.exists(zpath):
                req = urllib.request.Request(OMI_URL, headers={'User-Agent': 'Mozilla/5.0'})
                with urllib.request.urlopen(req, timeout=300) as r:
                    raw = r.read()
                with open(zpath, 'wb') as f:
                    f.write(raw)
            with zipfile.ZipFile(zpath) as z:
                d = pd.read_csv(z.open(z.namelist()[0]))
            d = d.rename(columns={d.columns[0]: 'date'})
            d['date'] = pd.to_datetime(d['date'].str[:10])
            d = d[d['Symbol'].isin(OMI_SYMBOLS)][['date', 'Symbol'] + OMI_COLS].rename(columns={'Symbol': 'symbol'})
            path = os.path.join(cache, OMI_FILE)
            d.sort_values(['symbol', 'date']).to_csv(path, index=False, float_format='%.6g')
        _CACHE['omi'] = pd.read_csv(path, parse_dates=['date'])
    d = _CACHE['omi']
    if symbols is not None:
        d = d[d['symbol'].isin([symbols] if isinstance(symbols, str) else list(symbols))]
    return d.copy()


def read_reference_rate(currency='EUR', start='2005-07-01', end=END):
    """Official BNR reference rate (RON per unit of currency), from the public yearly XML archives."""
    key = (currency, start, end)
    if key in _CACHE:
        return _CACHE[key]
    rows = []
    for y in range(max(int(start[:4]), 2005), int(end[:4]) + 1):
        url = f'https://curs.bnr.ro/files/xml/years/nbrfxrates{y}.xml'
        xml = urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent': 'Mozilla'}),
                                     timeout=60).read().decode()
        for d, body in re.findall(r'<Cube date="([\d-]+)">(.*?)</Cube>', xml, re.S):
            m = re.search(rf'<Rate currency="{currency}"(?: multiplier="\d+")?>([\d.]+)</Rate>', body)
            if m:
                rows.append((d, float(m.group(1))))
    s = pd.DataFrame(rows, columns=['date', 'close']).drop_duplicates('date').set_index('date')['close']
    s.index = pd.to_datetime(s.index)
    _CACHE[key] = s.sort_index().loc[start:end]
    return _CACHE[key]


def load_close(name, start=None, end=END, field=None):
    """Daily closing price of a named series, cleaned with the course conventions."""
    symbol, label, group, field0, start0 = SERIES[name]
    start, field = start or start0, field or field0
    if symbol.startswith('REF:'):
        return read_reference_rate(symbol.split(':')[1], start=start, end=end).rename(name)
    t = read_market(symbol)
    s = (t[field] if field in t.columns else t['close']).loc[start:end]
    s = pd.to_numeric(s, errors='coerce').dropna()
    if group != 'Yield':
        s = s[s > 0]
    if group != 'Crypto':
        s = s[s.index.dayofweek < 5]          # no weekend quotes
        s = s[s.diff() != 0]                  # no holidays filled with the previous price
    s.index.name = 'date'
    return s.rename(name)


def load_ohlc(name, start=None, end=END):
    """Open, high, low, close (and volume) of a series with OHLC data (range-based volatility estimators)."""
    symbol, _, group, _, start0 = SERIES[name]
    t = read_market(symbol).loc[start or start0:end]
    cols = [c for c in ('open', 'high', 'low', 'close', 'volume') if c in t.columns]
    t = t[cols].dropna()
    if group != 'Crypto':
        t = t[t.index.dayofweek < 5]
    return t[(t[['open', 'high', 'low', 'close']] > 0).all(axis=1)] if 'open' in cols else t


def log_returns(name, start=None, end=END, pct=True):
    """Daily log returns r_t = ln P_t - ln P_{t-1} (in % by default), on the series' own calendar."""
    r = np.log(load_close(name, start, end)).diff().dropna()
    return (100 * r if pct else r).rename(name)


def simple_returns(name, start=None, end=END, pct=True):
    """Daily simple returns R_t = P_t / P_{t-1} - 1 (in % by default)."""
    r = load_close(name, start, end).pct_change().dropna()
    return (100 * r if pct else r).rename(name)


def load_panel(names, start=None, end=END, kind='close'):
    """Several series aligned on date: kind = 'close' or 'returns' (NaN where a market does not trade)."""
    f = load_close if kind == 'close' else log_returns
    return pd.concat([f(n, start, end) for n in names], axis=1)


def periods_per_year(r):
    """Average number of observations per calendar year (the actual frequency of the series)."""
    years = (r.index[-1] - r.index[0]).days / 365.25
    return len(r) / years


def read_fred(ids, start=None, end=None):
    """FRED series (St. Louis Fed) from the public fredgraph CSV, e.g. read_fred('UNRATE') or
    read_fred(['GDPC1', 'CPIAUCSL']). Returns a Series (one id) or a DataFrame (several ids)."""
    one = isinstance(ids, str)
    ids = [ids] if one else list(ids)
    out = []
    for sid in ids:
        key = ('fred', sid)
        if key not in _CACHE:
            url = f'https://fred.stlouisfed.org/graph/fredgraph.csv?id={sid}'
            t = pd.read_csv(url, index_col=0, parse_dates=True, na_values='.')
            _CACHE[key] = pd.to_numeric(t.iloc[:, 0], errors='coerce').rename(sid)
        out.append(_CACHE[key])
    df = pd.concat(out, axis=1).loc[start:end]
    df.index.name = 'date'
    return df.iloc[:, 0].dropna() if one else df


def _eurostat_period(p):
    p = str(p)
    if '-Q' in p:
        y, q = p.split('-Q')
        return pd.Timestamp(int(y), 3 * int(q) - 2, 1)
    if '-' in p and len(p) == 7:
        return pd.Timestamp(p + '-01')
    if len(p) == 4:
        return pd.Timestamp(int(p), 1, 1)
    return pd.Timestamp(p)


def read_eurostat(dataset, key, start=None):
    """One Eurostat series from the public SDMX 2.1 API (SDMX-CSV), e.g. Romanian quarterly real GDP,
    seasonally and calendar adjusted, chain-linked volumes (2010) in million EUR:
        read_eurostat('namq_10_gdp', 'Q.CLV10_MEUR.SCA.B1GQ.RO')
    `key` is the dot-separated series key of the dataset (dimensions in the order of the dataset)."""
    url = (f'https://ec.europa.eu/eurostat/api/dissemination/sdmx/2.1/data/{dataset}/{key}?format=SDMX-CSV'
           + (f'&startPeriod={start}' if start else ''))
    ck = ('eurostat', url)
    if ck not in _CACHE:
        t = pd.read_csv(url)
        s = pd.Series(pd.to_numeric(t['OBS_VALUE'], errors='coerce').values,
                      index=[_eurostat_period(p) for p in t['TIME_PERIOD']], name=f'{dataset}:{key}')
        s.index.name = 'date'
        _CACHE[ck] = s.sort_index().dropna()
    return _CACHE[ck]


def load_statsmodels(name):
    """Classic data sets shipped with statsmodels (offline): 'sunspots' (yearly, 1700-2008), 'co2' (weekly,
    Mauna Loa, 1958-2001), 'macrodata' (US quarterly macro, 1959-2009), 'nile' (yearly Nile flow, 1871-1970)."""
    import statsmodels.api as sm
    if name == 'sunspots':
        d = sm.datasets.sunspots.load_pandas().data
        s = d.set_index(pd.to_datetime(d['YEAR'].astype(int).astype(str)))['SUNACTIVITY']
        s.index.name = 'date'
        return s.rename('sunspots')
    if name == 'co2':
        return sm.datasets.co2.load_pandas().data['co2'].rename('co2')
    if name == 'macrodata':
        d = sm.datasets.macrodata.load_pandas().data
        d.index = pd.period_range('1959Q1', periods=len(d), freq='Q').to_timestamp()
        d.index.name = 'date'
        return d
    if name == 'nile':
        d = sm.datasets.nile.load_pandas().data
        s = d.set_index(pd.to_datetime(d['year'].astype(int).astype(str)))['volume']
        s.index.name = 'date'
        return s.rename('nile')
    raise KeyError(f'unknown statsmodels data set: {name}')

## Analysis

In [ ]:
import io

# the cached indicator series: this folder, the chapter folder of a local copy, or GitHub (read_cached)

HERE = next((p for p in ('.', '../../Quantlets/Ch_16', '../Quantlets/Ch_16', 'Quantlets/Ch_16') if os.path.exists(os.path.join(p, 'ch16_ci_ssec.csv'))), '.')

PROCS = 1

SEED = 2026

END = '2026-09-18'

SAMPLES = {'sp500': ('GSPC.INDX', False, '1990-01-01', '2004-12-31', 'S&P 500, 1990-2004'), 'ndx': ('NDX.INDX', False, '1990-01-01', '2004-12-31', 'Nasdaq 100, 1990-2004'), 'ssec': ('SSEC.INDX', False, '2010-01-01', '2018-12-31', 'Shanghai Composite, 2010-2018'), 'btc': ('BTC-USD.CC', True, '2014-09-17', '2023-12-31', 'Bitcoin, 2014-2023'), 'bet': ('BET', False, '2000-01-01', '2012-12-31', 'BET, 2000-2012')}

PEAKS = {'sp500': [('1999-06-01', '2000-12-31')], 'ndx': [('1999-06-01', '2000-12-31')], 'ssec': [('2015-01-01', '2015-12-31')], 'btc': [('2017-06-01', '2018-03-31'), ('2021-01-01', '2021-06-30')], 'bet': [('2007-01-01', '2007-12-31')]}

COLORS = {'sp500': st.MainBlue, 'ndx': st.Teal, 'ssec': st.Orange, 'btc': st.Amber, 'bet': st.Crimson}

SHADE = '#DCE6F2'

CRASH, HORIZON = 0.2, 182

RO_END = '2026-03-31'

TAU = 52

_MEM = {}

CSV_RAW = 'https://raw.githubusercontent.com/danpele/Advanced-Time-Series/main/Quantlets/Ch_16/'

EVAL = {'sp500': ('GSPC.INDX', False, '1990-01-01', '1993-01-01'), 'btc': ('BTC-USD.CC', True, '2011-01-01', '2014-01-01')}

EXCLUDE = ('.GBOND', 'VIX', 'USDT', 'USDC', 'DAI', 'EURRON.FOREX', 'BTBETRETF', 'IBIT', 'ETHA')

LPPLS_SEARCH = {'m': (0.0, 1.0), 'w': (1.0, 50.0), 'tc_frac': (0.0, 0.3333333333333333), 'damping_min': 1.0}

LPPLS_FILTER = {'m': (0.01, 0.99), 'w': (2.0, 25.0), 'tc_frac': (0.0, 0.2), 'osc_min': 2.5, 'rel_err_max': 0.15, 'lomb_alpha': 0.1, 'ar1_alpha': 0.1}

WINDOWS = [750, 725, 700, 675, 650, 625, 600, 575, 550, 525, 500, 475, 450, 425, 400, 375, 350, 325, 300, 275, 250, 225, 200, 175, 150, 125, 100, 75, 50]

def _cums(y):
    """Cumulative sums for the regression Delta y_t = a + delta y_{t-1} (rows t = 1..n); y is (n+1,) or (R, n+1)."""
    y = np.atleast_2d(np.asarray(y, float))
    x, z = y[:, :-1], np.diff(y, axis=1)
    pad = lambda a: np.concatenate([np.zeros((a.shape[0], 1)), np.cumsum(a, axis=1)], axis=1)   # noqa: E731
    return pad(x), pad(x * x), pad(z), pad(z * z), pad(x * z)


def _adf_end(C, e, s):
    """t-statistic of delta on the windows [s, e] (s: vector of first rows, e: last row), for every replication."""
    Sx, Sxx, Sz, Szz, Sxz = (c[:, e + 1][:, None] - c[:, s] for c in C)
    n = (e + 1 - s).astype(float)
    den = n * Sxx - Sx ** 2
    b = (n * Sxz - Sx * Sz) / den
    a = (Sz - b * Sx) / n
    ssr = np.maximum(Szz - a * Sz - b * Sxz, 1e-300)
    return b / np.sqrt(ssr / (n - 2) * n / den)


def adf_window(y):
    """Right-tailed ADF on one window: intercept a, slope delta, its standard error and the t-statistic."""
    y = np.asarray(y, float)
    x, z = y[:-1], np.diff(y)
    X = np.column_stack([np.ones_like(x), x])
    beta, *_ = np.linalg.lstsq(X, z, rcond=None)
    e = z - X @ beta
    s2 = e @ e / (len(z) - 2)
    se = np.sqrt(s2 * np.linalg.inv(X.T @ X)[1, 1])
    return dict(a=float(beta[0]), delta=float(beta[1]), se=float(se), t=float(beta[1] / se), n=len(z))


def min_window(T, r0=None):
    """Smallest window of Phillips, Shi and Yu (2015): r0 = 0.01 + 1.8 / sqrt(T); w0 = floor(r0 T)."""
    r0 = 0.01 + 1.8 / np.sqrt(T) if r0 is None else r0
    return int(np.floor(r0 * T)), float(r0)


def _lag_design(y, k):
    """Rows t = k+1..n of the ADF regression with k lagged differences: z_t = Delta y_t,
    x_t = (1, y_{t-1}, Delta y_{t-1}, ..., Delta y_{t-k}); y is (R, n+1)."""
    dy = np.diff(y, axis=1)
    n = dy.shape[1]
    cols = [np.ones_like(dy[:, k:]), y[:, k:n]] + [dy[:, k - j:n - j] for j in range(1, k + 1)]
    return np.stack(cols, axis=2), dy[:, k:]


def _bsadf_lags(Y, w0, k, chunk=16):
    """BSADF and forward sequences with k lagged differences (cumulated cross-products, batched solves); entries
    before row k + w0 are NaN; the output has n columns like the k = 0 version."""
    R, n = Y.shape[0], Y.shape[1] - 1
    bs, fw = np.full((R, n), np.nan), np.full((R, n), np.nan)
    for i in range(0, R, chunk):
        X, z = _lag_design(Y[i:i + chunk], k)
        m, p = X.shape[1], X.shape[2]
        pad = lambda a: np.concatenate([np.zeros((a.shape[0], 1) + a.shape[2:]), np.cumsum(a, axis=1)], axis=1)   # noqa: E731
        CXX = pad(np.einsum('rti,rtj->rtij', X, X))
        CXz = pad(X * z[:, :, None])
        Czz = pad(z * z)
        for e in range(w0 - 1, m):
            s_ = np.arange(0, e - w0 + 2)
            XX = CXX[:, e + 1][:, None] - CXX[:, s_]
            Xz = CXz[:, e + 1][:, None] - CXz[:, s_]
            zz = Czz[:, e + 1][:, None] - Czz[:, s_]
            inv = np.linalg.inv(XX)
            b = np.einsum('rsij,rsj->rsi', inv, Xz)
            ssr = np.maximum(zz - np.einsum('rsi,rsi->rs', b, Xz), 1e-300)
            nn = (e + 1 - s_)[None, :]
            t = b[..., 1] / np.sqrt(ssr / (nn - p) * inv[..., 1, 1])
            bs[i:i + chunk, e + k] = t.max(axis=1)
            fw[i:i + chunk, e + k] = t[:, 0]
    return bs, fw


def psy(y, r0=None, w0=None, k=0):
    """ADF (whole sample), SADF (sup of forward-expanding ADF), GSADF (sup over start and end points), the BSADF
    sequence (sup over start points for each end point) and the start point that attains it (k = 0); k > 0 adds k
    lagged differences to every window regression."""
    y = np.asarray(y, float)
    n = len(y) - 1
    if w0 is None:
        w0, r0 = min_window(n, r0)
    if k:
        bs, fw = _bsadf_lags(y[None, :], w0, k)
        return dict(adf=float(fw[0, -1]), sadf=float(np.nanmax(fw)), gsadf=float(np.nanmax(bs)), bsadf=bs[0], fwd=fw[0],
                    w0=int(w0), r0=r0, n=n, k=k)
    C = _cums(y)
    bsadf, fwd, arg = np.full(n, np.nan), np.full(n, np.nan), np.full(n, -1)
    for e in range(w0 - 1, n):
        stat = _adf_end(C, e, np.arange(0, e - w0 + 2))[0]
        bsadf[e], fwd[e], arg[e] = stat.max(), stat[0], int(np.argmax(stat))
    return dict(adf=float(fwd[-1]), sadf=float(np.nanmax(fwd)), gsadf=float(np.nanmax(bsadf)), bsadf=bsadf, fwd=fwd,
                start=arg, w0=int(w0), r0=r0, n=n, k=0)


def bsadf_paths(Y, w0, chunk=64, k=0):
    """BSADF and forward (SADF) sequences for several paths (rows of Y), vectorised over start points and paths."""
    Y = np.atleast_2d(np.asarray(Y, float))
    if k:
        return _bsadf_lags(Y, w0, k)
    R, n = Y.shape[0], Y.shape[1] - 1
    bs, fw = np.full((R, n), np.nan), np.full((R, n), np.nan)
    for i in range(0, R, chunk):
        C = _cums(Y[i:i + chunk])
        for e in range(w0 - 1, n):
            stat = _adf_end(C, e, np.arange(0, e - w0 + 2))
            bs[i:i + chunk, e] = stat.max(axis=1)
            fw[i:i + chunk, e] = stat[:, 0]
    return bs, fw


def bic_lag(y, kmax):
    """Lag order of the null model Delta y_t = mu + sum_j phi_j Delta y_{t-j} + e_t chosen by BIC (common sample)."""
    dy = np.diff(np.asarray(y, float))
    z = dy[kmax:]
    best = (np.inf, 0)
    for k in range(kmax + 1):
        X = np.column_stack([np.ones_like(z)] + [dy[kmax - j:len(dy) - j] for j in range(1, k + 1)])
        e = z - X @ np.linalg.lstsq(X, z, rcond=None)[0]
        bic = len(z) * np.log(e @ e / len(z)) + (k + 1) * np.log(len(z))
        best = min(best, (bic, k))
    return best[1]


def null_paths(T, R, rng, sig=None):
    """Random walk with a weak drift, y_t = T^(-1) + y_{t-1} + sig_t e_t (PSY 2015); sig: optional volatility path."""
    e = rng.standard_normal((R, T)) * (1.0 if sig is None else np.asarray(sig)[None, :])
    return np.concatenate([np.zeros((R, 1)), np.cumsum(1.0 / T + e, axis=1)], axis=1)


def psy_cv(T, w0, R=2000, seed=2026, probs=(0.90, 0.95, 0.99)):
    """Monte Carlo critical values under the null: quantiles of ADF, SADF and GSADF, the pointwise 95% quantile of
    BSADF at each end point, and the null draws."""
    rng = np.random.default_rng(seed)
    bs, fw = bsadf_paths(null_paths(T, R, rng), w0)
    q = lambda a: {f'{int(round(100 * p))}': float(np.quantile(a, p)) for p in probs}   # noqa: E731
    return dict(adf=q(fw[:, -1]), sadf=q(np.nanmax(fw, axis=1)), gsadf=q(np.nanmax(bs, axis=1)),
                bsadf95=np.nanquantile(bs, 0.95, axis=0), draws=dict(gsadf=np.nanmax(bs, axis=1), sadf=np.nanmax(fw, axis=1),
                                                                    adf=fw[:, -1]), bs=bs)


def wild_cv(y, w0, B=499, seed=2026, tau=None, k=0):
    """Wild bootstrap of Phillips and Shi (2020): residuals of the null model Delta y_t = mu + sum_j phi_j Delta y_{t-j}
    + e_t times Rademacher signs, Delta y*_t = sum_j phi_j Delta y*_{t-j} + w_t e_t, cumulated into paths with a unit
    root and the volatility pattern of the data. Returns the pointwise 95% BSADF sequence, the 95% quantile of the
    sample maximum (GSADF: family-wise over the whole sample) and, when tau is given, for every end point the 95%
    quantile of the maximum of BSADF over the last tau end points (family-wise over a moving window)."""
    rng = np.random.default_rng(seed)
    dy = np.diff(np.asarray(y, float))
    if k:
        X = np.column_stack([np.ones(len(dy) - k)] + [dy[k - j:len(dy) - j] for j in range(1, k + 1)])
        phi = np.linalg.lstsq(X, dy[k:], rcond=None)[0]
        e = dy[k:] - X @ phi
        W = rng.choice([-1.0, 1.0], size=(B, len(e)))
        D = np.zeros((B, len(dy)))
        D[:, :k] = dy[:k] - dy.mean()
        for t in range(k, len(dy)):
            D[:, t] = sum(phi[j] * D[:, t - j] for j in range(1, k + 1)) + W[:, t - k] * e[t - k]
    else:
        e = dy - dy.mean()
        D = rng.choice([-1.0, 1.0], size=(B, len(e))) * e
    Y = np.concatenate([np.zeros((B, 1)), np.cumsum(D, axis=1)], axis=1)
    bs, fw = bsadf_paths(Y, w0, k=k)
    out = dict(bsadf95=np.nanquantile(bs, 0.95, axis=0), gsadf95=float(np.quantile(np.nanmax(bs, axis=1), 0.95)),
               sadf95=float(np.quantile(np.nanmax(fw, axis=1), 0.95)), draws=np.nanmax(bs, axis=1))
    if tau:
        roll = pd.DataFrame(bs.T).rolling(tau, min_periods=1).max().values.T
        out['fwer95'] = np.nanquantile(roll, 0.95, axis=0)
    return out


def episodes(stat, cv, index, min_len):
    """Date-stamping: runs of at least min_len consecutive end points with stat > cv (cv scalar or sequence).
    Returns (start, end, length); the start is dated at the first exceedance, the end at the last one."""
    stat = np.asarray(stat, float)
    cv = np.broadcast_to(np.asarray(cv, float), stat.shape)
    above = (stat > cv) & ~np.isnan(stat) & ~np.isnan(cv)
    out, i, n = [], 0, len(above)
    while i < n:
        if above[i]:
            j = i
            while j + 1 < n and above[j + 1]:
                j += 1
            if j - i + 1 >= min_len:
                out.append((index[i], index[j], int(j - i + 1)))
            i = j + 1
        else:
            i += 1
    return out


def weekly(s):
    """Weekly log price: last close of each week (Friday)."""
    return np.log(s.resample('W-FRI').last().dropna())


def blanchard_watson(T=400, r=0.02, pi=0.98, b0=1.0, sd=0.3, seed=13):
    """Blanchard-Watson bubble: with probability pi it survives and grows by (1 + r)/pi, otherwise it bursts to
    noise; E_t[b_{t+1}] = (1 + r) b_t."""
    rng = np.random.default_rng(seed)
    b = np.empty(T)
    b[0] = b0
    for t in range(1, T):
        eps = sd * rng.standard_normal()
        b[t] = (1 + r) / pi * b[t - 1] + eps if rng.random() < pi else eps
    return b


def evans_price(T=400, r=0.05, alpha=1.0, delta=0.5, pi=0.85, tau=0.05, mu=0.0373, sd=0.3967, d0=1.3, scale=20.0,
                seed=11):
    """Price = present-value fundamental + scale * Evans (1991) bubble. Dividends: random walk with drift mu, so the
    fundamental is P^f_t = mu (1 + r) / r^2 + D_t / r. The bubble grows at 1 + r while B <= alpha; above alpha it
    survives with probability pi (growing faster) or collapses to delta; u_t lognormal with mean 1."""
    rng = np.random.default_rng(seed)
    D = d0 + np.cumsum(mu + sd * rng.standard_normal(T))
    pf = mu * (1 + r) / r ** 2 + D / r
    u = np.exp(rng.normal(-0.5 * tau ** 2, tau, T))
    B = np.empty(T)
    B[0] = delta
    for t in range(1, T):
        if B[t - 1] <= alpha:
            B[t] = (1 + r) * B[t - 1] * u[t]
        else:
            theta = rng.random() < pi
            B[t] = (delta + (1 + r) / pi * (B[t - 1] - delta / (1 + r)) * theta) * u[t]
    return pf, scale * B


def bubble_collapse(T, te, tf, rho, y0=100.0, sd=1.0, rng=None):
    """Random walk up to te, explosive y_t = rho y_{t-1} + e_t on (te, tf], collapse at tf + 1 back to the level of
    te, random walk afterwards (the bubble-and-collapse process used for date-stamping experiments)."""
    rng = rng or np.random.default_rng()
    e = sd * rng.standard_normal(T + 1)
    y = np.empty(T + 1)
    y[0] = y0
    for t in range(1, T + 1):
        if t <= te:
            y[t] = y[t - 1] + e[t]
        elif t <= tf:
            y[t] = rho * y[t - 1] + e[t]
        elif t == tf + 1:
            y[t] = y[te] + e[t]
        else:
            y[t] = y[t - 1] + e[t]
    return y


def ar1_explosive(n, rho, R, rng, dist='normal'):
    """R samples y_1..y_n of y_t = rho y_{t-1} + u_t, y_0 = 0; u_t standard normal or centred exponential."""
    u = rng.standard_normal((R, n)) if dist == 'normal' else rng.exponential(1.0, (R, n)) - 1.0
    y = np.zeros((R, n + 1))
    for t in range(1, n + 1):
        y[:, t] = rho * y[:, t - 1] + u[:, t - 1]
    return y


def ols_rho(y):
    """OLS of y_t on y_{t-1} without intercept, row by row."""
    x, z = y[:, :-1], y[:, 1:]
    return (x * z).sum(axis=1) / (x * x).sum(axis=1)


def csw_constant(alpha=0.05):
    """One-sided boundary-crossing constant: P{W(s) >= sqrt((s + 1)(a^2 + ln(s + 1))) for some s >= 0}
    = 1 - Phi(a) + a phi(a) (Robbins-Siegmund); solve for a at level alpha."""
    f = lambda a: 1 - stats.norm.cdf(a) + a * stats.norm.pdf(a) - alpha   # noqa: E731
    return float(optimize.brentq(f, 0.5, 6.0))


def cusum_monitor(r, n, alpha=0.05):
    """CUSUM monitoring of the mean of returns r after a training sample of n observations:
    S_t = sum_{j=n+1}^t (r_j - mean_n) / (sigma_n sqrt(n)) with the Chu-Stinchcombe-White boundary
    b_t = sqrt(((t - n)/n)(t/n)(a^2 + ln(t/(t - n)))), which S crosses under the null with probability alpha
    (one-sided, over an infinite horizon). Returns S, b and the first crossing (index in r, or None)."""
    r = np.asarray(r, float)
    mu, sig = r[:n].mean(), r[:n].std(ddof=1)
    S = np.cumsum(r[n:] - mu) / (sig * np.sqrt(n))
    t = np.arange(n + 1, len(r) + 1)
    a = csw_constant(alpha)
    b = np.sqrt((t - n) / n * t / n * (a ** 2 + np.log(t / (t - n))))
    hit = np.nonzero(S > b)[0]
    return S, b, (int(n + hit[0]) if len(hit) else None)


def yrs(idx):
    """Dates as decimal years (the time unit of the LPPLS fits)."""
    idx = pd.DatetimeIndex(idx)
    return np.asarray(idx.year + (idx.dayofyear - 1) / 365.25, float)


def todate(x):
    """Decimal year -> date."""
    y = int(np.floor(x))
    return pd.Timestamp(y, 1, 1) + pd.Timedelta(days=float((x - y) * 365.25))


def lppl_design(t, tc, m, w):
    dt = np.maximum(tc - np.asarray(t, float), 1e-9)
    f = dt ** m
    lg = np.log(dt)
    return np.column_stack([np.ones_like(dt), f, f * np.cos(w * lg), f * np.sin(w * lg)])


def lppl_linear(t, y, tc, m, w):
    """Linear step: for given (tc, m, omega), A, B, C1, C2 by OLS and the sum of squared residuals."""
    X = lppl_design(t, tc, m, w)
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    res = y - X @ beta
    return beta, float(res @ res)


def _damping(m, w, beta):
    C = np.hypot(beta[..., 2], beta[..., 3])
    return m * np.abs(beta[..., 1]) / (w * np.maximum(C, 1e-300))


def _grid(t, y, tcs, ms, ws):
    """SSR of the linear step on a grid of (tc, m, omega), vectorised."""
    TC, M, W = (a.ravel() for a in np.meshgrid(tcs, ms, ws, indexing='ij'))
    dt = np.maximum(TC[:, None] - t[None, :], 1e-9)
    f = dt ** M[:, None]
    lg = np.log(dt)
    X = np.stack([np.ones_like(f), f, f * np.cos(W[:, None] * lg), f * np.sin(W[:, None] * lg)], axis=2)
    XtX = np.einsum('gni,gnj->gij', X, X) + 1e-10 * np.eye(4)
    Xty = np.einsum('gni,n->gi', X, y)
    beta = np.linalg.solve(XtX, Xty[..., None])[..., 0]
    ssr = y @ y - np.einsum('gi,gi->g', beta, Xty)
    return TC, M, W, beta, ssr


def lppl_fit(t, y, search=LPPLS_SEARCH, grid=(8, 8, 16), tc_fixed=None):
    """Nonlinear step: minimum SSR over (tc, m, omega) in the search space (damping >= 1): a grid, then Nelder-Mead
    from the best grid point. With tc_fixed, only (m, omega) are free (profile of the critical time)."""
    t, y = np.asarray(t, float), np.asarray(y, float)
    t1, t2 = t[0], t[-1]
    D = t2 - t1
    lo = np.array([t2 + search['tc_frac'][0] * D + 1e-6, search['m'][0] + 1e-3, search['w'][0]])
    hi = np.array([t2 + search['tc_frac'][1] * D, search['m'][1] - 1e-3, search['w'][1]])
    if tc_fixed is not None:
        lo[0] = hi[0] = tc_fixed
    tcs = np.array([tc_fixed]) if tc_fixed is not None else np.linspace(lo[0], hi[0], grid[0])
    TC, M, W, beta, ssr = _grid(t, y, tcs, np.linspace(lo[1], hi[1], grid[1]), np.linspace(lo[2], hi[2], grid[2]))
    ok = _damping(M, W, beta) >= search['damping_min']
    if not ok.any():
        return None
    k = int(np.argmin(np.where(ok, ssr, np.inf)))
    x0 = np.array([TC[k], M[k], W[k]])
    free = slice(1, 3) if tc_fixed is not None else slice(0, 3)

    def obj(p):
        q = x0.copy()
        q[free] = p
        q = np.clip(q, lo, hi)
        b, sr = lppl_linear(t, y, *q)
        return sr if _damping(q[1], q[2], b) >= search['damping_min'] else sr + 1e6
    r = optimize.minimize(obj, x0[free], method='Nelder-Mead', options=dict(xatol=1e-5, fatol=1e-10, maxiter=400))
    if r.fun < 1e6:
        x0[free] = r.x
        x0 = np.clip(x0, lo, hi)
    tc, m, w = x0
    (A, B, C1, C2), s = lppl_linear(t, y, tc, m, w)
    C = np.hypot(C1, C2)
    yhat = lppl_design(t, tc, m, w) @ np.array([A, B, C1, C2])
    return dict(tc=float(tc), m=float(m), w=float(w), A=float(A), B=float(B), C1=float(C1), C2=float(C2), C=float(C),
                ssr=float(s), damping=float(m * abs(B) / (w * C)) if C > 0 else np.inf,
                osc=float(w / np.pi * np.log((tc - t1) / (tc - t2))) if tc > t2 else 0.0,
                rel_err=float(np.max(np.abs(np.exp(yhat) - np.exp(y)) / np.exp(y))), t1=float(t1), t2=float(t2),
                n=len(t), _t=t, _y=y, _yhat=yhat)


def lomb_pvalue(fit, wmin=2.0, wmax=25.0, nw=200):
    """Lomb test of the detrended residual (tc - t)^(-m) (ln p - A - B (tc - t)^m) against ln(tc - t)."""
    from scipy.signal import lombscargle
    t, y = fit['_t'], fit['_y']
    dt = np.maximum(fit['tc'] - t, 1e-9)
    r = dt ** (-fit['m']) * (y - fit['A'] - fit['B'] * dt ** fit['m'])
    x = np.log(dt)
    r = r - r.mean()
    ws = np.linspace(wmin, wmax, nw)
    p = lombscargle(x, r, ws) / r.var()
    return float(1 - (1 - np.exp(-p.max())) ** min(nw, len(r)))


def ar1_pass(fit, alpha):
    """The residual ln p_hat - ln p is stationary: the Dickey-Fuller and Phillips-Perron tests both reject a unit
    root at level alpha (as in TSA, Chapter 13)."""
    from statsmodels.tsa.stattools import adfuller
    from arch.unitroot import PhillipsPerron
    e = fit['_yhat'] - fit['_y']
    return bool(adfuller(e, maxlag=0, autolag=None, regression='c')[1] < alpha and PhillipsPerron(e, trend='c').pvalue < alpha)


def lppl_conditions(fit, flt=LPPLS_FILTER, search=LPPLS_SEARCH):
    """Filter of Shu and Zhu (2020), eq. (12), for a positive bubble (B < 0); the Lomb test only when the parameter
    conditions and the relative error pass, the residual unit-root test only when Lomb passes."""
    if fit is None:
        return None
    D = fit['t2'] - fit['t1']
    c = dict(B=fit['B'] < 0, m=flt['m'][0] <= fit['m'] <= flt['m'][1], w=flt['w'][0] <= fit['w'] <= flt['w'][1],
             tc=fit['t2'] + flt['tc_frac'][0] * D <= fit['tc'] <= fit['t2'] + flt['tc_frac'][1] * D,
             osc=fit['osc'] >= flt['osc_min'], damping=fit['damping'] >= search['damping_min'],
             rel_err=fit['rel_err'] <= flt['rel_err_max'])
    if all(c.values()):
        c['lomb'] = lomb_pvalue(fit) <= flt['lomb_alpha']
        c['ar1'] = ar1_pass(fit, flt['ar1_alpha']) if c['lomb'] else False
    else:
        c['lomb'] = c['ar1'] = None
    return c


def qualified(c):
    return c is not None and all(v is True for v in c.values())


def ci_point(args):
    """LPPLS confidence indicator at one end point: the share of windows whose fit passes the whole filter, and the
    median critical time of the qualified fits."""
    t, y, i2, windows = args
    full, tcs = [], []
    for L in windows:
        i1 = i2 - L
        if i1 < 0:
            continue
        f = lppl_fit(t[i1:i2 + 1], y[i1:i2 + 1])
        q = qualified(lppl_conditions(f))
        full.append(q)
        if q:
            tcs.append(f['tc'])
    return (float(np.mean(full)) if full else np.nan, float(np.median(tcs)) if tcs else np.nan)


def confidence_series(s, start, end=None, step=5, windows=WINDOWS, procs=1):
    """LPPLS confidence indicator (positive bubbles) at the end points t2 in [start, end], every `step` observations;
    procs > 1 uses a process pool."""
    t, y = yrs(s.index), np.log(s.values)
    i0 = s.index.searchsorted(pd.Timestamp(start))
    i1 = len(s) if end is None else s.index.searchsorted(pd.Timestamp(end), 'right')
    jobs = [(t, y, int(i), windows) for i in range(i0, i1, step)]
    if procs == 1:
        res = [ci_point(j) for j in jobs]
    else:
        from multiprocessing import get_context
        with get_context('fork').Pool(procs) as pool:
            res = pool.map(ci_point, jobs, chunksize=4)
    return pd.DataFrame(res, index=s.index[[j[2] for j in jobs]], columns=['ci', 'tc_median'])


def tc_profile(t, y, tcs):
    """Profile log-likelihood of the critical time (Gaussian errors): for each tc, (m, omega) and the linear
    parameters are concentrated out; l(tc) = -(n/2) ln(SSR(tc)/n)."""
    out = []
    for tc in tcs:
        f = lppl_fit(t, y, search=dict(LPPLS_SEARCH, tc_frac=(0.0, 10.0)), tc_fixed=tc)
        out.append(-0.5 * len(t) * np.log(f['ssr'] / len(t)) if f is not None else np.nan)
    return np.array(out)


def future_fall(s, horizon=182):
    """For each date t: min_{t < u <= t + horizon days} p_u / p_t - 1 (NaN when the horizon is not observed)."""
    p, d = s.values, s.index
    out = np.full(len(s), np.nan)
    j = 0
    for i in range(len(s)):
        lim = d[i] + pd.Timedelta(days=horizon)
        if lim > d[-1]:
            break
        j = max(j, i + 1)
        while j < len(s) and d[j] <= lim:
            j += 1
        out[i] = p[i + 1:j].min() / p[i] - 1 if j > i + 1 else np.nan
    return pd.Series(out, index=d)


def auc(score, event):
    """Area under the ROC curve: P(score of an event date > score of a non-event date), ties counted 1/2."""
    score, event = np.asarray(score, float), np.asarray(event, bool)
    r = stats.rankdata(score)
    n1, n0 = event.sum(), (~event).sum()
    return float((r[event].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)) if n1 and n0 else np.nan


def roc(score, event):
    """ROC curve: false-alarm rate and hit rate for every threshold."""
    score, event = np.asarray(score, float), np.asarray(event, bool)
    o = np.argsort(-score, kind='mergesort')
    e = event[o]
    tp, fp = np.cumsum(e), np.cumsum(~e)
    return np.r_[0, fp / max(fp[-1], 1)], np.r_[0, tp / max(tp[-1], 1)]


def block_bootstrap_auc(score, event, block=26, B=999, seed=2026):
    """Moving-block bootstrap of the AUC (blocks of consecutive evaluation dates keep the overlap of the horizons)."""
    rng = np.random.default_rng(seed)
    score, event = np.asarray(score, float), np.asarray(event, bool)
    n = len(score)
    k = int(np.ceil(n / block))
    out = []
    for _ in range(B):
        idx = (rng.integers(0, n - block + 1, k)[:, None] + np.arange(block)[None, :]).ravel()[:n]
        out.append(auc(score[idx], event[idx]))
    return np.array(out)


def alarm_table(alarm, event):
    """Hit rate, false-alarm rate, precision and base rate of a 0/1 alarm against a 0/1 event."""
    a, e = np.asarray(alarm, bool), np.asarray(event, bool)
    return dict(hit=float((a & e).sum() / max(e.sum(), 1)), false_alarm=float((a & ~e).sum() / max((~e).sum(), 1)),
                precision=float((a & e).sum() / max(a.sum(), 1)) if a.sum() else np.nan, base=float(e.mean()),
                n_alarm=int(a.sum()), n=int(len(a)))


def manifest():
    """The list of saved series (data/manifest.csv): symbol, name, number of observations, first and last day."""
    p = os.path.join(MARKET_DIR, '..', 'manifest.csv') if MARKET_DIR else ''
    src = p if p and os.path.exists(p) else REPO_RAW.replace('data/market/', 'data/manifest.csv')
    return pd.read_csv(src)


def read_cached(name):
    """A cached indicator series of the chapter: local copy (Quantlets/Ch_16) or the ATS repository on GitHub."""
    for p in (os.path.join(HERE, name), CSV_RAW + name):
        try:
            return pd.read_csv(p, index_col=0, parse_dates=True)
        except Exception:
            pass
    return None


def save(name, save_it=True):
    st.check_no_grey(plt.gcf())
    if save_it:
        st.save_fig(name)
    else:
        plt.show()
        plt.close()


def d2s(d):
    return pd.Timestamp(d).strftime('%Y-%m-%d')


def prices(symbol, crypto=False, field='close'):
    """Daily close with the course conventions (weekdays only and no holiday-filled closes, except crypto)."""
    t = read_market(symbol)
    s = pd.to_numeric(t[field] if field in t.columns else t['close'], errors='coerce').dropna()
    s = s[s > 0].loc[:END]
    if not crypto:
        s = s[s.index.dayofweek < 5]
        s = s[s.diff() != 0]
    return s.rename(symbol)


def min_len(T):
    """Minimum duration of a dated episode: ceil(ln T) end points."""
    return int(np.ceil(np.log(T)))


def _pool(func, jobs):
    if PROCS == 1 or len(jobs) == 1:
        return [func(j) for j in jobs]
    from multiprocessing import get_context
    with get_context('fork').Pool(min(PROCS, len(jobs))) as pool:
        return pool.map(func, jobs, chunksize=1)


def fig_rational(save_it=True, R=2000):
    """Blanchard-Watson bubble; Evans bubble on a present-value fundamental with BSADF against its 95% critical value."""
    T = 400
    rng = np.random.default_rng(SEED)
    fund = 20 + np.cumsum(0.3 * rng.standard_normal(T))
    b = blanchard_watson(T)
    pf, B = evans_price(T)
    p = pf + B
    o = psy(p)
    cv = psy_cv(o['n'], o['w0'], R=R, seed=SEED)
    L = min_len(o['n'])
    ep = episodes(o['bsadf'], cv['bsadf95'], np.arange(1, o['n'] + 1), L)
    fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
    axes[0].plot(fund + b, color=st.IDAred, label='Blanchard-Watson: fundamental + bubble')
    axes[0].plot(fund, color=st.MainBlue, label='fundamental value')
    axes[0].set_title('Blanchard-Watson bubble')
    axes[1].plot(p, color=st.IDAred, label='Evans: fundamental + bubble')
    axes[1].plot(pf, color=st.MainBlue, label='_fundamental')
    axes[1].set_title('Evans (1991) bubble')
    axes[2].plot(np.arange(1, o['n'] + 1), o['bsadf'], color=st.MainBlue, label='BSADF of the Evans price')
    axes[2].plot(np.arange(1, o['n'] + 1), cv['bsadf95'], color=st.IDAred, ls='--', label='95% critical value')
    for a_, b_, _ in ep:
        for ax in axes[1:]:
            ax.axvspan(a_, b_, color=st.Orange, alpha=0.18, lw=0)
    axes[2].set_title('Date-stamping the Evans bubble')
    for ax in axes:
        ax.set_xlabel('period')
    axes[0].set_ylabel('price')
    st.fig_legend_bottom(fig, ncol=5, y=0.02)
    fig.tight_layout(rect=(0, 0.07, 1, 1))
    save('ats_ch16_rational', save_it)
    bursts = int(np.sum((B[1:] < 0.6 * 20) & (B[:-1] > 1.0 * 20)))
    return dict(T=o['n'], w0=o['w0'], r0=o['r0'], adf=o['adf'], sadf=o['sadf'], gsadf=o['gsadf'],
                cv={k: cv[k] for k in ('adf', 'sadf', 'gsadf')}, n_ep=len(ep), episodes=[(int(a), int(b_), int(n)) for a, b_, n in ep],
                L=L, collapses=bursts, bw_dur=1 / (1 - 0.98), R=R)

## Run

In [ ]:
print(fig_rational(save_it=False, R=500))

![ats_ch16_rational](./ats_ch16_rational.png)

Output: `ats_ch16_rational.pdf`